# Advanced Python Descriptor Problems — `__set_name__` and Beyond

**One self-contained, executable notebook · 18 problems with full solutions and executable checks · Python 3.10+ · standard library only**

### Learning goals
- Explain exactly when `__set_name__(owner, name)` runs, and when it does **not** run.
- Implement robust, reusable validating data descriptors with per-instance storage.
- Predict data/non-data descriptor precedence, class-level access, inheritance and deletion.
- Handle `__slots__`, weak-reference storage, dynamic class construction, descriptor reuse and dataclasses.
- Integrate descriptors into metaclasses, introspection and atomic multi-field updates.

**How to use:** Run all cells from top to bottom. Each problem provides a challenge, a design explanation, a complete reference implementation and an assertion-based test cell. Before reading each solution, try implementing the challenge on your own. A passing test cell prints `Problem NN: PASS`. No external packages or network access are required.

**Source relationship:** This notebook expands the supplied lesson on `__set_name__`, a `ValidString` descriptor, naming collisions, and the preview of data versus non-data descriptor precedence. It introduces additional problems beyond the supplied material. One precision: `__set_name__` is automatically invoked when a class **is created** for descriptors found directly in its namespace, not when a descriptor object itself is instantiated; inherited descriptors are not automatically rebound.

In [1]:
import sys
import weakref
import gc
from dataclasses import dataclass, InitVar
from inspect import getattr_static

assert sys.version_info >= (3, 10), 'Use Python 3.10 or newer for this notebook.'
print('Python:', sys.version.split()[0])
print('Ready: standard-library-only descriptor exercises')

Python: 3.13.7
Ready: standard-library-only descriptor exercises


---
## Problem 01 — Class-creation lifecycle and exact invocation order
**Difficulty:** Advanced

### Challenge
Implement a descriptor that records constructor calls and `__set_name__` calls. Show that constructing a descriptor alone does not call `__set_name__`, that binding it in a newly created class does, that class-body attribute order matters, and that looking up a class attribute does not re-bind it. Include an explicit `__get__` implementation so class access is inspectable.

**Think first:** Distinguish descriptor **object construction** (`__init__`) from owner-class construction. In `__get__`, handle `instance is None` explicitly.

### Solution and reasoning
Python invokes `__set_name__` from class creation machinery on attributes in the *new class namespace* whose objects implement that method. It runs after the class namespace has been prepared and before `__init_subclass__` for a new subclass. A descriptor object may receive this callback more than once if explicitly placed in multiple class namespaces; this is why a blanket claim of "once per descriptor instance" is inaccurate.

In [2]:
events_01 = []
class Tracing01:
    def __init__(self, label):
        self.label = label
        events_01.append(('init', label))

    def __set_name__(self, owner, name):
        events_01.append(('set_name', owner.__name__, name))
        self.owner = owner
        self.name = name

    def __get__(self, instance, owner=None):
        return self if instance is None else f'{self.name}@{type(instance).__name__}'

free_01 = Tracing01('free')
assert events_01 == [('init', 'free')]

class Lifecycle01:
    alpha = Tracing01('a')
    beta = Tracing01('b')

print('Lifecycle events:', events_01)

Lifecycle events: [('init', 'free'), ('init', 'a'), ('init', 'b'), ('set_name', 'Lifecycle01', 'alpha'), ('set_name', 'Lifecycle01', 'beta')]


In [3]:
assert events_01 == [
    ('init', 'free'), ('init', 'a'), ('init', 'b'),
    ('set_name', 'Lifecycle01', 'alpha'),
    ('set_name', 'Lifecycle01', 'beta'),
]
assert Lifecycle01.alpha is Lifecycle01.__dict__['alpha']
assert Lifecycle01().beta == 'beta@Lifecycle01'
assert len(events_01) == 5  # accesses did not call __set_name__
print('Problem 01: PASS')

Problem 01: PASS


---
## Problem 02 — Production-style validated string field
**Difficulty:** Advanced

### Challenge
Build `ValidatedString02(min_length, max_length=None, allow_blank=False)` and a `Customer02` class. Require exact `str` inputs (no implicit string conversion); reject strings shorter or longer than the configured limits; keep state separate for each instance; expose the descriptor on class access; distinguish an unset field from a field whose value is `None`; support deletion. Use an instance dictionary key equal to the public attribute name.

**Think first:** Validate constructor configuration. `instance.__dict__[self.name]` avoids recursing through `setattr(instance, self.name, ...)`. An unset field should raise `AttributeError`, not silently return `None`.

### Solution and reasoning
Because `ValidatedString02` defines `__set__` and `__delete__`, it is a **data descriptor**. It takes precedence over a same-named key in the instance dictionary; directly accessing `instance.__dict__` inside `__get__` is intentional, not recursion. Use `TypeError` for wrong types and `ValueError` for value constraints. Direct writes to `__dict__` remain a potential validation bypass; this design does not promise tamper-proof storage.

In [4]:
class ValidatedString02:
    def __init__(self, min_length=1, max_length=None, *, allow_blank=False):
        if type(min_length) is not int or min_length < 0:
            raise ValueError('min_length must be a nonnegative integer')
        if max_length is not None and (type(max_length) is not int or max_length < min_length):
            raise ValueError('max_length must be an integer >= min_length')
        self.min_length = min_length
        self.max_length = max_length
        self.allow_blank = allow_blank

    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if type(value) is not str:
            raise TypeError(f'{self.name} must be a str')
        if not self.allow_blank and not value.strip():
            raise ValueError(f'{self.name} cannot be blank')
        if len(value) < self.min_length:
            raise ValueError(f'{self.name} needs at least {self.min_length} characters')
        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(f'{self.name} allows at most {self.max_length} characters')
        return value

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{type(instance).__name__}.{self.name} is unset') from None

    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} is already unset') from None

class Customer02:
    first_name = ValidatedString02(2, 20)
    last_name = ValidatedString02(2, 30)

    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

In [5]:
a02 = Customer02('Ada', 'Lovelace')
b02 = Customer02('Grace', 'Hopper')
assert a02.first_name == 'Ada' and b02.first_name == 'Grace'
assert a02.__dict__ == {'first_name':'Ada', 'last_name':'Lovelace'}
assert Customer02.first_name is Customer02.__dict__['first_name']
for bad, error in [(42, TypeError), (' ', ValueError), ('X', ValueError), ('x'*21, ValueError)]:
    try:
        a02.first_name = bad
    except error as exc:
        assert 'first_name' in str(exc)
    else:
        raise AssertionError((bad, 'should have failed'))
assert a02.first_name == 'Ada'  # failed validation never mutates state
del a02.first_name
try:
    _ = a02.first_name
except AttributeError:
    pass
else:
    raise AssertionError('Reading an unset field must fail')
try:
    ValidatedString02(3, 2)
except ValueError:
    pass
else:
    raise AssertionError('Invalid configuration must fail')
print('Problem 02: PASS')

Problem 02: PASS


---
## Problem 03 — Predict the exact attribute lookup precedence
**Difficulty:** Advanced

### Challenge
Create one non-data descriptor (only `__get__`) and one data descriptor (`__get__` and `__set__`) that record all reads. Arrange for both to share their own public names with instance dictionary entries. Demonstrate which lookup wins, and explain why direct dictionary access is not the same operation as regular attribute lookup.

**Think first:** Data descriptor > instance dictionary > non-data descriptor > other class attributes (simplified usual instance-lookup order). Class access often receives `instance=None`.

### Solution and reasoning
The presence of `__set__` *or* `__delete__` makes a descriptor data-descriptor-like for lookup precedence. Writing `obj.__dict__['data']` bypasses the descriptor's setter but `obj.data` still calls the data descriptor's getter. A non-data descriptor can be shadowed by the same-named instance key.

In [6]:
reads_03 = []
class NonData03:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        reads_03.append(('nondata', self.name))
        return 'from non-data descriptor'

class Data03:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        reads_03.append(('data', self.name))
        return 'from data descriptor'
    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class Lookup03:
    nondata = NonData03()
    data = Data03()

obj03 = Lookup03()
obj03.__dict__.update(nondata='dictionary nondata', data='dictionary data')
print('regular access:', obj03.nondata, '|', obj03.data)
print('direct dictionary:', obj03.__dict__)

regular access: dictionary nondata | from data descriptor
direct dictionary: {'nondata': 'dictionary nondata', 'data': 'dictionary data'}


In [7]:
assert obj03.nondata == 'dictionary nondata'
assert obj03.data == 'from data descriptor'
assert reads_03 == [('data','data'), ('data','data')]
assert Lookup03.nondata is Lookup03.__dict__['nondata']
assert Lookup03.data is Lookup03.__dict__['data']
del obj03.__dict__['nondata']
assert obj03.nondata == 'from non-data descriptor'
assert reads_03[-1] == ('nondata','nondata')
print('Problem 03: PASS')

Problem 03: PASS


---
## Problem 04 — Eliminate accidental backing-key collisions
**Difficulty:** Advanced

### Challenge
The source lesson uses `_' + property_name` as a backing key and demonstrates overwriting unrelated instance state. Reproduce that bug, then implement a same-name, data-descriptor design that preserves `_name` and validates accesses. Explain the remaining caveat of manual dictionary mutation.

**Think first:** The descriptor is stored on the class, but can intentionally store validated instance values under `instance.__dict__[name]` because data descriptors win normal reads.

### Solution and reasoning
A private-looking prefix is only a naming convention, not guaranteed storage isolation. Same-name dictionary storage removes collisions with independently used `_name`, but users can still deliberately change `__dict__['name']`, bypassing the setter. Use a descriptor plus an encapsulation strategy if adversarial mutation is a concern.

In [8]:
class Fragile04:
    def __set_name__(self, owner, name):
        self.key = '_' + name
    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__.get(self.key)
    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

class FragilePerson04:
    name = Fragile04()

fragile04 = FragilePerson04()
fragile04._name = 'unrelated private state'
fragile04.name = 'Ada'
print('Original backing key after assignment:', fragile04._name)

class SaferPerson04:
    name = ValidatedString02(1)

safe04 = SaferPerson04()
safe04._name = 'unrelated private state'
safe04.name = 'Ada'
print('Same-name storage:', safe04.__dict__)

Original backing key after assignment: Ada
Same-name storage: {'_name': 'unrelated private state', 'name': 'Ada'}


In [9]:
assert fragile04._name == 'Ada'  # demonstrates real data loss
assert safe04._name == 'unrelated private state'
assert safe04.__dict__['name'] == safe04.name == 'Ada'
safe04.__dict__['name'] = 123  # deliberately bypasses validation
assert safe04.name == 123  # show limitation instead of pretending this is impossible
safe04.name = 'Grace'      # setter restores the validated state
assert safe04.name == 'Grace'
print('Problem 04: PASS')

Problem 04: PASS


---
## Problem 05 — Inheritance: definition-time owner versus access-time owner
**Difficulty:** Advanced

### Challenge
Trace descriptor creation in a base class and a subclass. Verify that an inherited descriptor does **not** get a new automatic `__set_name__` call, but its `__get__` receives the accessing subclass as `owner`. Then override that field with a new descriptor to show the new callback.

**Think first:** Store the owner from `__set_name__` separately from the owner argument passed to `__get__`. Access through both an instance and class.

### Solution and reasoning
`__set_name__` is limited to bindings in the newly created class's own namespace. Inherited descriptor objects keep their original definition-time owner/name; runtime `owner` arguments may refer to a subclass. These two meanings should not be conflated.

In [10]:
events_05 = []
class OwnerAware05:
    def __set_name__(self, owner, name):
        self.defining_owner = owner
        self.public_name = name
        events_05.append((owner.__name__, name))
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return (self.defining_owner.__name__, owner.__name__, self.public_name)

class Base05:
    field = OwnerAware05()

class Child05(Base05):
    pass

class Override05(Base05):
    field = OwnerAware05()

print('callbacks:', events_05)
print('inherited access:', Child05().field)

callbacks: [('Base05', 'field'), ('Override05', 'field')]
inherited access: ('Base05', 'Child05', 'field')


In [11]:
assert events_05 == [('Base05','field'), ('Override05','field')]
assert Base05.field is Child05.field
assert Base05().field == ('Base05','Base05','field')
assert Child05().field == ('Base05','Child05','field')
assert Override05().field == ('Override05','Override05','field')
assert Override05.field is not Base05.field
print('Problem 05: PASS')

Problem 05: PASS


---
## Problem 06 — Dynamic attachment versus dynamic class creation
**Difficulty:** Advanced

### Challenge
Install a name-dependent descriptor with `setattr(SomeClass, 'field', descriptor)`, and explain why it is not automatically initialized. Explicitly call `__set_name__` afterward. Contrast this with constructing a class through `type('Name', (object,), {'field': descriptor})`, which triggers the callback automatically.

**Think first:** `setattr` modifies an existing class; `type(name, bases, namespace)` creates a new class. Never assume late assignment reruns class creation.

### Solution and reasoning
Late descriptor attachment needs a manual callback if the descriptor uses it to configure itself. Call `descriptor.__set_name__(Owner, 'field')` deliberately and only once for that specific binding. This is not required for a new class created via `type`.

In [12]:
class DynamicField06:
    def __init__(self):
        self.bindings = []
    def __set_name__(self, owner, name):
        self.name = name
        self.bindings.append((owner.__name__, name))
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return self.name

class Existing06:
    pass

late06 = DynamicField06()
setattr(Existing06, 'field', late06)
assert late06.bindings == []
late06.__set_name__(Existing06, 'field')
new06 = DynamicField06()
Created06 = type('Created06', (object,), {'field': new06})
print('late:', late06.bindings, 'new class:', new06.bindings)

late: [('Existing06', 'field')] new class: [('Created06', 'field')]


In [13]:
assert Existing06().field == 'field'
assert late06.bindings == [('Existing06','field')]
assert Created06().field == 'field'
assert new06.bindings == [('Created06','field')]
print('Problem 06: PASS')

Problem 06: PASS


---
## Problem 07 — `__slots__`-compatible descriptor storage
**Difficulty:** Expert

### Challenge
An instance with `__slots__` and no `__dict__` breaks dictionary-backed descriptors. Define a validating positive-integer descriptor that derives a separate slot name from `__set_name__`. Declare that slot in the owner class. Use the slot's built-in member descriptor as the underlying storage mechanism; show that unrelated instances never share state.

**Think first:** Call `object.__setattr__(instance, private_slot, value)` and read that private slot with `object.__getattribute__`. Using `self.name` for storage would recurse; declaring a slot with the *same name* as a class descriptor conflicts.

### Solution and reasoning
`__slots__` names are class-level member descriptors. A separate `_quantity` slot provides per-instance storage without `__dict__`. The public validating descriptor `quantity` and storage slot `_quantity` must have distinct names. The slot name should be declared explicitly: `__set_name__` cannot retroactively add a slot after class creation.

In [14]:
class PositiveSlot07:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = '_' + name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return object.__getattribute__(instance, self.storage_name)
        except AttributeError:
            raise AttributeError(f'{self.name} has not been initialized') from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be exactly int')
        if value <= 0:
            raise ValueError(f'{self.name} must be positive')
        object.__setattr__(instance, self.storage_name, value)

class Inventory07:
    __slots__ = ('_quantity',)
    quantity = PositiveSlot07()
    def __init__(self, quantity):
        self.quantity = quantity

left07, right07 = Inventory07(3), Inventory07(12)
print(left07.quantity, right07.quantity)

3 12


In [15]:
assert not hasattr(left07, '__dict__')
assert (left07.quantity, right07.quantity) == (3,12)
left07.quantity = 8
assert (left07.quantity, right07.quantity) == (8,12)
for bad, exception in [(True, TypeError), (0, ValueError), ('10', TypeError)]:
    try:
        left07.quantity = bad
    except exception:
        pass
    else:
        raise AssertionError(f'Invalid value {bad!r} was accepted')
assert left07.quantity == 8
print('Problem 07: PASS')

Problem 07: PASS


---
## Problem 08 — Weak-key storage and object lifetime
**Difficulty:** Expert

### Challenge
Implement a descriptor with `weakref.WeakKeyDictionary` to store values without using the instance dictionary. Demonstrate that garbage collection removes an instance's entry. State its limits: weak-reference support, hashability, weak-key equality semantics, and avoiding strong references to the key from the stored value.

**Think first:** Give a slotted owner `__weakref__` and use default identity-based hashing. Do not store the instance itself as the dictionary value.

### Solution and reasoning
A `WeakKeyDictionary` automatically removes entries when keys die. However, not every instance can be weakly referenced (especially some slotted classes), and unhashable instances cannot be weak keys. It can use the key object's equality/hash behavior rather than guaranteeing identity isolation; use default object equality/hash here. If a value strongly refers back to its key, that reference may prevent collection.

In [16]:
class WeakField08:
    def __init__(self):
        self._values = weakref.WeakKeyDictionary()
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None
    def __set__(self, instance, value):
        if type(value) is not int or value < 0:
            raise ValueError(f'{self.name} must be a nonnegative integer')
        self._values[instance] = value
    def __delete__(self, instance):
        try:
            del self._values[instance]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None

class SlottedUser08:
    __slots__ = ('__weakref__',)
    score = WeakField08()

first08 = SlottedUser08()
second08 = SlottedUser08()
first08.score, second08.score = 10, 20
weak_first08 = weakref.ref(first08)
print('Entry count before deletion:', len(SlottedUser08.score._values))

Entry count before deletion: 2


In [17]:
assert not hasattr(first08, '__dict__')
assert len(SlottedUser08.score._values) == 2
assert (first08.score, second08.score) == (10,20)
del first08
gc.collect()
assert weak_first08() is None
assert len(SlottedUser08.score._values) == 1
assert second08.score == 20
print('Problem 08: PASS')

Problem 08: PASS


---
## Problem 09 — Detect accidental reuse of a stateful descriptor
**Difficulty:** Expert

### Challenge
A descriptor that stores one `self.name` becomes ambiguous if the same descriptor object is bound to two attributes. Write a fail-fast `SingleBinding09` that rejects a second, different owner/name pair. Prove a same-named second binding is rejected, and inspect how Python reports the exception raised by `__set_name__` during `type(...)` class creation across Python versions.

**Think first:** Some versions wrap this error as `RuntimeError` with the original exception in `__cause__`; others surface the `ValueError` directly. Normalize both paths in the test.

### Solution and reasoning
Reuse is sometimes possible with a descriptor designed for multiple bindings, but a single `name` field is not such a design. Reject ambiguous reuse early. During class creation, exception presentation can vary by Python version; inspect the original `ValueError` directly or through an exception's `__cause__` instead of assuming a wrapper type.

In [18]:
class SingleBinding09:
    def __init__(self):
        self.binding = None
    def __set_name__(self, owner, name):
        if self.binding is not None:
            raise ValueError(f'descriptor already bound to {self.binding[0].__name__}.{self.binding[1]}')
        self.binding = (owner, name)
    def __get__(self, instance, owner=None):
        return self if instance is None else self.binding[1]

shared09 = SingleBinding09()
First09 = type('First09', (), {'field': shared09})
try:
    Second09 = type('Second09', (), {'field': shared09})
except (RuntimeError, ValueError) as err09:
    binding_error09 = err09.__cause__ if err09.__cause__ is not None else err09
else:
    raise AssertionError('Descriptor reuse must be rejected')
print('Original binding error:', type(binding_error09).__name__, str(binding_error09))

Original binding error: ValueError descriptor already bound to First09.field


In [19]:
assert First09().field == 'field'
assert isinstance(binding_error09, ValueError)
assert 'First09.field' in str(binding_error09)
assert shared09.binding == (First09, 'field')
print('Problem 09: PASS')

Problem 09: PASS


---
## Problem 10 — Descriptor registration via a metaclass
**Difficulty:** Expert

### Challenge
Implement a metaclass that produces `__fields__`: an immutable mapping from every effective public field name to its `RegisteredField10` descriptor. Include inherited fields in MRO order, support subclass overrides (including overriding a descriptor with a plain attribute), and avoid changing a parent's registry.

**Think first:** Create the class with `super().__new__` *before* inspecting newly bound descriptors; their `__set_name__` methods have then been called. Filter an inherited field if a subclass namespace replaces it with something else.

### Solution and reasoning
Using `__set_name__` to store each descriptor's original binding and metaclass `__new__` to build a class registry keeps configuration centralized. Build a *new* mapping per class. Copy the direct base's already-effective registry, then process every local namespace name: a local plain attribute removes any inherited field of the same name, while a local descriptor adds or replaces it. This example covers single inheritance; for multiple inheritance, carefully match actual MRO semantics.

In [20]:
from types import MappingProxyType

class RegisteredField10:
    def __set_name__(self, owner, name):
        self.name = name
        self.defining_owner = owner
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)
    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class FieldMeta10(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        fields = {}
        # This exercise uses single inheritance; inherited registries are copied.
        for base in bases:
            fields.update(getattr(base, '__fields__', {}))
        for attr_name, attr in namespace.items():
            if isinstance(attr, RegisteredField10):
                fields[attr_name] = attr
            else:
                fields.pop(attr_name, None)
        cls.__fields__ = MappingProxyType(fields)
        return cls

class Record10(metaclass=FieldMeta10):
    name = RegisteredField10()
    age = RegisteredField10()

class Employee10(Record10):
    age = 'not a field anymore'
    salary = RegisteredField10()

class Manager10(Employee10):
    age = RegisteredField10()

print('Record fields:', tuple(Record10.__fields__))
print('Employee fields:', tuple(Employee10.__fields__))
print('Manager fields:', tuple(Manager10.__fields__))

Record fields: ('name', 'age')
Employee fields: ('name', 'salary')
Manager fields: ('name', 'salary', 'age')


In [21]:
assert tuple(Record10.__fields__) == ('name','age')
assert tuple(Employee10.__fields__) == ('name','salary')
assert tuple(Manager10.__fields__) == ('name','salary','age')
assert Employee10.__fields__['name'] is Record10.name
assert Manager10.__fields__['age'].defining_owner is Manager10
try:
    Employee10.__fields__['bad'] = RegisteredField10()
except TypeError:
    pass
else:
    raise AssertionError('Registry must be read-only')
assert tuple(Record10.__fields__) == ('name','age')
print('Problem 10: PASS')

Problem 10: PASS


---
## Problem 11 — Composed validators and a subtle `bool` trap
**Difficulty:** Expert

### Challenge
Create a generic integer descriptor accepting a list of callable validators `(value) -> bool`, each paired with an error explanation. Configure a transaction record with quantity and unit_price, and verify errors name the failing field. Reject `True` even though `isinstance(True, int)` is true.

**Think first:** Prefer `type(value) is int` when the business requirement says *exact integer*. Keep descriptor configuration immutable after construction to reduce accidental cross-owner modifications.

### Solution and reasoning
The validators are distinct, composable constraints, but their callbacks should be deterministic and side-effect-free for trustworthy validation. `bool` subclasses `int`, so `isinstance(True, int)` is insufficient for exact-integer validation.

In [22]:
class IntegerRules11:
    def __init__(self, *rules):
        self.rules = tuple(rules)
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None
    def validate(self, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an exact int')
        for predicate, description in self.rules:
            if not predicate(value):
                raise ValueError(f'{self.name}: {description}')
        return value
    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)

class LineItem11:
    quantity = IntegerRules11((lambda n: n > 0, 'must be positive'),
                              (lambda n: n <= 10_000, 'exceeds capacity'))
    unit_price_cents = IntegerRules11((lambda n: n >= 0, 'cannot be negative'))
    def __init__(self, quantity, unit_price_cents):
        self.quantity = quantity
        self.unit_price_cents = unit_price_cents
    @property
    def subtotal_cents(self):
        return self.quantity * self.unit_price_cents

item11 = LineItem11(3, 250)
print('subtotal in cents:', item11.subtotal_cents)

subtotal in cents: 750


In [23]:
assert item11.subtotal_cents == 750
for field, bad, expected in [('quantity',True,TypeError),
                             ('quantity',0,ValueError),
                             ('quantity',10001,ValueError),
                             ('unit_price_cents',-1,ValueError)]:
    try:
        setattr(item11, field, bad)
    except expected as exc:
        assert field in str(exc)
    else:
        raise AssertionError((field,bad))
assert (item11.quantity,item11.unit_price_cents) == (3,250)
assert isinstance(True,int) and type(True) is not int
print('Problem 11: PASS')

Problem 11: PASS


---
## Problem 12 — Build a caching non-data descriptor
**Difficulty:** Expert

### Challenge
Implement `CachedOnce12`, a decorator-style descriptor that computes a value the first time it is accessed on an instance, then stores the result under its own public attribute name in `instance.__dict__`. Confirm future reads shadow the descriptor; deleting that dictionary entry forces recomputation. State why adding `__set__` to this descriptor would change its behavior.

**Think first:** This intentionally uses non-data descriptor precedence in the opposite direction from the validated-string exercise. The result can be manually overridden using normal instance assignment.

### Solution and reasoning
A non-data descriptor's getter runs only while an instance has no overriding same-name dictionary key. Once it caches under that name, instance lookup returns the cached value directly. If the descriptor implemented `__set__` or `__delete__`, it would become a data descriptor and *normal lookup would no longer bypass its getter*; caching would then need an explicit check within `__get__`.

In [24]:
class CachedOnce12:
    def __init__(self, function):
        self.function = function
        self.__doc__ = function.__doc__
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        value = self.function(instance)
        instance.__dict__[self.name] = value
        return value

class Report12:
    def __init__(self, data):
        self.data = list(data)
        self.compute_count = 0
    @CachedOnce12
    def total(self):
        """Compute an expensive total once."""
        self.compute_count += 1
        return sum(self.data)

report12 = Report12([5,7,9])
print('first:',report12.total, 'second:',report12.total)
print('computed:', report12.compute_count)

first: 21 second: 21
computed: 1


In [25]:
assert report12.total == 21
assert report12.compute_count == 1
assert Report12.total.name == 'total'
report12.data.append(10)
assert report12.total == 21  # stale cache is intentional until invalidation
del report12.__dict__['total']
assert report12.total == 31
assert report12.compute_count == 2
report12.total = 999  # non-data descriptor allows shadowing
assert report12.total == 999 and report12.compute_count == 2
print('Problem 12: PASS')

Problem 12: PASS


---
## Problem 13 — Read-only data descriptor versus dictionary shadowing
**Difficulty:** Advanced

### Challenge
Construct `ReadOnly13`, whose setter always raises `AttributeError`. Confirm that an instance dictionary entry with the descriptor's name cannot shadow normal reads; direct class assignment can still replace the descriptor itself, so do not claim it enforces universal immutability.

**Think first:** A setter that *raises* is still a setter for precedence. Keep the invariant on the descriptor instance instead of the owning instance.

### Solution and reasoning
Python's normal instance lookup prioritizes data descriptors even when they reject writes. This is a way to expose derived or constant data; it does not stop a caller from replacing the descriptor in the owner class or bypassing more general object protection mechanisms.

In [26]:
class ReadOnly13:
    def __init__(self, constant):
        self.constant = constant
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        return self if instance is None else self.constant
    def __set__(self, instance, value):
        raise AttributeError(f'{self.name} is read-only')

class Config13:
    environment = ReadOnly13('production')

config13 = Config13()
config13.__dict__['environment'] = 'development'
print('Normal access:', config13.environment)
print('Raw dictionary:', config13.__dict__['environment'])

Normal access: production
Raw dictionary: development


In [27]:
assert config13.environment == 'production'
assert config13.__dict__['environment'] == 'development'
assert Config13.environment is Config13.__dict__['environment']
try:
    config13.environment = 'staging'
except AttributeError as exc:
    assert 'environment' in str(exc)
else:
    raise AssertionError('Read-only descriptor accepted assignment')
print('Problem 13: PASS')

Problem 13: PASS


---
## Problem 14 — Dataclass initialization through a descriptor
**Difficulty:** Expert

### Challenge
Integrate a descriptor with a dataclass without accidentally turning the descriptor itself into a field default. Accept a constructor-only `InitVar` named `initial_balance` and route it through a validating `balance` descriptor in `__post_init__`. Show the repr/field distinction and reject invalid initial balances.

**Think first:** `InitVar` parameters reach `__post_init__` but are not stored as ordinary dataclass fields. The descriptor can be declared as a `ClassVar` so dataclasses will not treat it as an instance field.

### Solution and reasoning
When dataclass fields and descriptors overlap, the generated `__init__` may interact with descriptors in surprising ways. An explicit `ClassVar` descriptor plus an `InitVar` constructor input and `__post_init__` gives intentional validation and a clean dataclass representation.

In [28]:
from typing import ClassVar
from dataclasses import fields

class Nonnegative14:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None
    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be integer cents')
        if value < 0:
            raise ValueError(f'{self.name} cannot be negative')
        instance.__dict__[self.name] = value

@dataclass
class Wallet14:
    label: str
    initial_balance: InitVar[int]
    balance: ClassVar[Nonnegative14] = Nonnegative14()
    def __post_init__(self, initial_balance):
        self.balance = initial_balance

wallet14 = Wallet14('travel', 1500)
print('repr:', wallet14)
print('fields:', [field.name for field in fields(Wallet14)])
print('validated balance:', wallet14.balance)

repr: Wallet14(label='travel')
fields: ['label']
validated balance: 1500


In [29]:
assert wallet14.label == 'travel' and wallet14.balance == 1500
assert [field.name for field in fields(Wallet14)] == ['label']
assert Wallet14.balance.name == 'balance'
assert 'balance=' not in repr(wallet14)
for bad, exception in [(-1, ValueError), (True, TypeError)]:
    try:
        Wallet14('bad', bad)
    except exception:
        pass
    else:
        raise AssertionError(f'Invalid initial balance {bad!r} was accepted')
print('Problem 14: PASS')

Problem 14: PASS


---
## Problem 15 — Atomic multi-field validation and update
**Difficulty:** Expert

### Challenge
Implement an `update_fields` method for a small record that atomically validates all user-supplied changes *before* committing any of them. Reuse each descriptor's validation method; reject unknown names, avoid partial state changes if the second field fails, and support normal single-field assignment.

**Think first:** First produce a temporary dictionary of validated values; only after every check succeeds should you write to the instance dictionary. This is atomic with respect to validation errors in a single-threaded context, not a general concurrency transaction.

### Solution and reasoning
The descriptors still enforce ordinary assignments. The batch method intentionally writes directly to `__dict__` only after using each descriptor's own validation routine, ensuring no partially applied mutation on validation failures. For concurrent callers, use synchronization; a dictionary update alone is not a full transaction across external effects.

In [30]:
class NonnegativeInt15:
    def __set_name__(self, owner, name):
        self.name = name
    def validate(self, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an int')
        if value < 0:
            raise ValueError(f'{self.name} must be nonnegative')
        return value
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None
    def __set__(self, instance, value):
        instance.__dict__[self.name] = self.validate(value)

class Stock15:
    in_store = NonnegativeInt15()
    reserved = NonnegativeInt15()
    def __init__(self, in_store, reserved):
        self.update_fields(in_store=in_store, reserved=reserved)
    def update_fields(self, **changes):
        validated = {}
        for name, value in changes.items():
            descriptor = vars(type(self)).get(name)
            if not isinstance(descriptor, NonnegativeInt15):
                raise AttributeError(f'unknown validated field: {name}')
            validated[name] = descriptor.validate(value)
        self.__dict__.update(validated)

stock15 = Stock15(12,3)
stock15.update_fields(in_store=20, reserved=5)
print('after successful update:', stock15.__dict__)

after successful update: {'in_store': 20, 'reserved': 5}


In [31]:
assert (stock15.in_store,stock15.reserved) == (20,5)
for changes, exception in [({'in_store':2, 'reserved':-1}, ValueError),
                           ({'in_store':1, 'unknown':100}, AttributeError),
                           ({'reserved':True}, TypeError)]:
    original = stock15.__dict__.copy()
    try:
        stock15.update_fields(**changes)
    except exception:
        pass
    else:
        raise AssertionError(f'Invalid batch succeeded: {changes}')
    assert stock15.__dict__ == original  # all-or-nothing on validation failures
stock15.in_store = 100
assert stock15.in_store == 100
print('Problem 15: PASS')

Problem 15: PASS


---
## Problem 16 — Introspection without invoking descriptor getters
**Difficulty:** Expert

### Challenge
Use a descriptor whose `__get__` increments a counter. Demonstrate the difference between `getattr(instance, 'field')` and `inspect.getattr_static(instance, 'field')`, and show how to find the defining descriptor object in the class dictionary without running the getter. Include class access and the `instance=None` convention.

**Think first:** Introspection tools such as `getattr_static` deliberately avoid dynamic lookup and do not necessarily return the same *value* as `getattr`. They may return the raw descriptor.

### Solution and reasoning
`getattr_static` is useful in debugging, frameworks, and documentation generators when invoking arbitrary descriptor code is undesirable. It returns the raw descriptor in this example; it does not resolve properties or descriptors into their ordinary runtime values. `vars(Owner)['field']` is another way to access a locally defined descriptor without calling `__get__`.

In [32]:
class Logged16:
    def __init__(self):
        self.get_calls = 0
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        self.get_calls += 1
        return f'computed-{self.name}'

class Document16:
    field = Logged16()

doc16 = Document16()
raw16 = getattr_static(doc16, 'field')
before16 = raw16.get_calls
value16 = getattr(doc16, 'field')
print('Raw object:', type(raw16).__name__)
print('Runtime value:', value16, '| getter calls:', raw16.get_calls)

Raw object: Logged16
Runtime value: computed-field | getter calls: 1


In [33]:
assert raw16 is vars(Document16)['field']
assert raw16 is Document16.field
assert before16 == 0 and raw16.get_calls == 1
assert value16 == 'computed-field'
assert getattr_static(doc16, 'field') is raw16
assert raw16.get_calls == 1  # introspection avoided code execution
print('Problem 16: PASS')

Problem 16: PASS


---
## Problem 17 — Descriptor hook order versus `__init_subclass__`
**Difficulty:** Expert

### Challenge
Instrument `__set_name__`, a base class's `__init_subclass__`, and a metaclass's `__new__`. Prove the class-creation sequence for a newly defined subclass, and avoid assumptions about class-body execution being part of the descriptor callback.

**Think first:** The metaclass can log before `super().__new__` and after it. The default type construction performs `__set_name__`, then calls the parent's `__init_subclass__`, before returning to metaclass `__new__`.

### Solution and reasoning
The relevant order is: metaclass `__new__` entry → `type.__new__` calls local `__set_name__` → base `__init_subclass__` → metaclass `__new__` exit. Class-body execution already occurred before the metaclass receives its namespace. This order lets `__init_subclass__` inspect initialized descriptors.

In [34]:
order17 = []

class Meta17(type):
    def __new__(mcls, name, bases, namespace):
        order17.append(('meta-enter', name))
        cls = super().__new__(mcls, name, bases, namespace)
        order17.append(('meta-exit', name))
        return cls

class Base17(metaclass=Meta17):
    def __init_subclass__(cls, **kwargs):
        order17.append(('init-subclass', cls.__name__))
        # The descriptor is initialized before the hook runs.
        assert cls.__dict__['field'].name == 'field'
        super().__init_subclass__(**kwargs)

class Hook17:
    def __set_name__(self, owner, name):
        self.name = name
        order17.append(('set-name',owner.__name__,name))

order17.clear()  # ignore creation of Base17; measure only derived class
class Derived17(Base17):
    field = Hook17()

print('Events:', order17)

Events: [('meta-enter', 'Derived17'), ('set-name', 'Derived17', 'field'), ('init-subclass', 'Derived17'), ('meta-exit', 'Derived17')]


In [35]:
assert order17 == [
    ('meta-enter','Derived17'),
    ('set-name','Derived17','field'),
    ('init-subclass','Derived17'),
    ('meta-exit','Derived17'),
]
print('Problem 17: PASS')

Problem 17: PASS


---
## Problem 18 — Implement deletion-only precedence and full diagnostic matrix
**Difficulty:** Expert

### Challenge
Show that implementing `__delete__` (without `__set__`) also makes a descriptor data for attribute-lookup precedence. Put a conflicting key into the instance dictionary, verify descriptor reads win, invoke deletion, and explain why direct dictionary mutation is still possible. Finish with an explicit matrix of operations and their effects.

**Think first:** Data descriptor classification checks the presence of `__set__` **or** `__delete__`. An assignment may be rejected by the descriptor machinery even if the descriptor defines no custom setter.

### Solution and reasoning
This corner case is easy to overlook: `__delete__` is sufficient for data-descriptor precedence. In CPython's ordinary attribute assignment path, trying to assign to such an attribute without a setter raises `AttributeError` instead of silently inserting an instance key. Direct `__dict__` writes are separate operations and are not intercepted.

In [36]:
class DeleteOnly18:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, instance, owner=None):
        return self if instance is None else 'controlled by descriptor'
    def __delete__(self, instance):
        instance.__dict__.pop(self.name, None)

class DeletionCase18:
    token = DeleteOnly18()

case18 = DeletionCase18()
case18.__dict__['token'] = 'manual shadow value'
print('instance lookup:', case18.token)
print('raw dictionary:', case18.__dict__['token'])
del case18.token
print('after descriptor delete:', case18.__dict__)

instance lookup: controlled by descriptor
raw dictionary: manual shadow value
after descriptor delete: {}


In [37]:
assert case18.token == 'controlled by descriptor'
assert 'token' not in case18.__dict__
try:
    case18.token = 'disallowed'
except AttributeError:
    pass
else:
    raise AssertionError('Delete-only descriptor should reject normal assignment')
case18.__dict__['token'] = 'manual write'
assert case18.token == 'controlled by descriptor'
print('Problem 18: PASS')

Problem 18: PASS


---
# Consolidated conceptual answer key

| Question | Correct answer |
|---|---|
| Does descriptor object construction call `__set_name__`? | No. Class creation calls it for relevant objects in the new class namespace. |
| Does subclassing automatically call `__set_name__` on an inherited field? | No. It remains bound to its original defining class and name. |
| Does `setattr(ExistingClass, name, descriptor)` invoke `__set_name__`? | No. Call it manually when needed. |
| What happens on `Owner.field` access? | The descriptor's `__get__` conventionally returns itself when `instance is None`; custom implementations can choose another behavior. |
| Can a data descriptor store data at `obj.__dict__[public_name]`? | Yes. Data-descriptor reads still take precedence in ordinary attribute access. |
| Can direct `__dict__` mutation bypass validation? | Yes. Standard descriptors intercept ordinary attribute operations, not arbitrary dictionary edits. |
| Does a non-data descriptor lose to an instance key? | Yes, in normal instance attribute lookup. |
| Does defining `__delete__` without `__set__` create data-descriptor precedence? | Yes. |
| Can `__set_name__` add slots retroactively? | No. Slot declarations are resolved during class creation; reserve appropriate storage slots in the class body. |
| Can one descriptor instance be reused under arbitrary aliases safely? | Not if it stores just one owner/name; implement multi-binding support or reject reuse. |

### Best-practice checklist
1. Return the descriptor itself for class-level access when introspection is desired.
2. Raise `TypeError` for incompatible types, `ValueError` for constraints, and `AttributeError` for absent values where appropriate.
3. Keep per-instance values on instances or in carefully chosen external storage, never a single shared `self.value` on the descriptor.
4. Avoid `setattr(instance, self.name, value)` inside a descriptor setter: it commonly recurses into itself.
5. Document whether manual `__dict__` edits, weak references, or class-level monkeypatching are within the threat model.
6. Ensure tests cover multiple owner instances, invalid assignment without mutation, class-level access, inheritance, deletion, and the unset state.

**Further challenges (try independently):** build a class-registry metaclass with multiple-inheritance/MRO conflict tests; integrate a thread lock into batch updates; create a multi-owner descriptor that keeps distinct `(owner, name)` bindings; or implement a cached data descriptor with explicit invalidation and TTL. These are *extension prompts*, not silently claimed as solved above.

In [38]:
# Final execution marker: if Run All reaches this cell, every assertion above passed.
print('ALL 18 ADVANCED PROBLEMS PASSED')

ALL 18 ADVANCED PROBLEMS PASSED
